# 4.0.0 Collostructional analysis for "dark matter" mentions

Load the master-derived lexical mention source, configure collostruction thresholds and output paths, and keep the lexical branch analytically distinct from the balanced analysis corpus used in the main workflow.


In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
NOTEBOOKS_DIR = CODE_DIR / "notebooks"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

# Repo mode omits manuscript and Overleaf export surfaces.
def record_figure_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

def record_table_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

import json
import math
import os
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

from notebook_theme import CAT_LABELS, activate_theme, colors
from dm_term_normalization.normalization import (
    TARGET,
    pat,
    MAX_CTX_WORDS,
    MIN_CTX_WORDS,
    MIN_CENTER_SENT_WORDS,
    SOFT_CTX_WORDS,
    MAX_DM_PER_CTX,
    MAX_MATH_TOKENS_PER_CTX,
    canon_dm,
    iter_sentences_norm,
)
from dm_term_normalization.tfidf_preproc import preproc, tokenize_preproc_text

THEME_NAME = "paper"
theme = activate_theme(THEME_NAME)

BASE_RUN_ID = "analysis_corpus"
ANALYSIS_DATASET = "analysis_corpus"
LEXICAL_SOURCE_MODE = "analysis_corpus"
LEXICAL_SOURCE_MODE_LABELS = {
    "analysis_corpus": "balanced analysis_corpus bins",
    "master_derived": "master-derived mentions filtered to analysis bins",
}
if LEXICAL_SOURCE_MODE not in LEXICAL_SOURCE_MODE_LABELS:
    raise ValueError(f"Unsupported LEXICAL_SOURCE_MODE: {LEXICAL_SOURCE_MODE}")
LEXICAL_SOURCE_NAMESPACE = (
    ANALYSIS_DATASET
    if LEXICAL_SOURCE_MODE == ANALYSIS_DATASET
    else f"{ANALYSIS_DATASET}__{LEXICAL_SOURCE_MODE}"
)
CATEGORIES = ["A", "B"]
LABEL_SPACE = ["A", "B", "C"]

WINDOW_CHARS = 550
MIN_FREQ = 20
CELL_TOP_N = 30
CELL_WEIGHT_MIN_PROB = 0.10
FORCE_REBUILD_LEXICAL_SOURCE = True
FORCE_REBUILD_LEXICAL_CLASSIFIED = True
FORCE_REBUILD_LEXICAL_CORPUS = True
FORCE_REBUILD_COLLOC = True

MODEL_DIR = OUTPUTS_DIR / "analytical-results" / "models" / "scibert"
BEST_MODEL = MODEL_DIR / "scibert_classifier_3labs_best.pt"
THRESHOLDS_PATH = MODEL_DIR / "scibert_classifier_3labs_thresholds.json"
SCIBERT_LOCAL_DIR = LOCAL_DIR / "models" / "scibert_scivocab_uncased"
SCIBERT_BASE = str(SCIBERT_LOCAL_DIR) if SCIBERT_LOCAL_DIR.exists() else "allenai/scibert_scivocab_uncased"
MAX_LEN = 256
BATCH_SIZE = 128
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)

MASTER_RUN_ID = "master_corpus"
MASTER_PARQUET = LOCAL_DIR / "corpora" / "master_corpus" / "master_corpus.parquet"
MASTER_MANIFEST = LOCAL_DIR / "corpora" / "master_corpus" / "master_corpus_manifest.json"
BASE_CORPUS_ROOT = LOCAL_DIR / "corpora" / BASE_RUN_ID
BASE_CORPUS_DIR = BASE_CORPUS_ROOT / "bins"
BASE_CORPUS_MANIFEST = BASE_CORPUS_ROOT / "corpus_manifest.json"
CLASSIFIED_CORPUS_PATH = OUTPUTS_DIR / "analytical-results" / BASE_RUN_ID / "corpus" / "corpus_classified.jsonl"

if not MASTER_MANIFEST.exists():
    raise FileNotFoundError(f"Master corpus manifest not found: {MASTER_MANIFEST}")
with open(MASTER_MANIFEST, "r", encoding="utf-8") as f:
    master_manifest = json.load(f)

if not BASE_CORPUS_MANIFEST.exists():
    raise FileNotFoundError(f"Base corpus manifest not found: {BASE_CORPUS_MANIFEST}")
with open(BASE_CORPUS_MANIFEST, "r", encoding="utf-8") as f:
    base_corpus_manifest_preview = json.load(f)

BIN_LABELS_SOURCE = list(base_corpus_manifest_preview.get("written_bins") or base_corpus_manifest_preview.get("final_bin_counts", {}).keys())
if not BIN_LABELS_SOURCE:
    raise ValueError(f"Could not derive BIN_LABELS_SOURCE from {BASE_CORPUS_MANIFEST}")
print("bin_labels_source:", BIN_LABELS_SOURCE)

base_run_root = OUTPUTS_DIR / "analytical-results" / BASE_RUN_ID
colloc_root = base_run_root / "collostructional"
ANALYSIS_DATA_DIR = colloc_root / "data" / LEXICAL_SOURCE_NAMESPACE
COLLOC_PLOT_ROOT = colloc_root / "plots"
COLLOC_THEME_ROOT = COLLOC_PLOT_ROOT / THEME_NAME / LEXICAL_SOURCE_NAMESPACE
HEATMAP_PLOT_DIR = COLLOC_THEME_ROOT / "heatmaps"
TRAJECTORY_PLOT_DIR = COLLOC_THEME_ROOT / "trajectories"
BUBBLE_PLOT_DIR = COLLOC_THEME_ROOT / "bubble"
TIMELINE_PLOT_DIR = COLLOC_THEME_ROOT / "timelines"
LEXICAL_MENTIONS_FILE = ANALYSIS_DATA_DIR / "lexical_mentions.jsonl"
LEXICAL_SOURCE_MANIFEST = ANALYSIS_DATA_DIR / "lexical_source_manifest.json"
LEXICAL_CLASSIFIED_FILE = ANALYSIS_DATA_DIR / "lexical_classified.jsonl"
LEXICAL_CORPUS_RUN_ID = "lexical_master"
LEXICAL_CORPUS_DIR = colloc_root / "source" / LEXICAL_SOURCE_NAMESPACE
LEXICAL_CORPUS_MANIFEST = LEXICAL_CORPUS_DIR / "corpus_manifest.json"

COLLOC_DATA_KEY = "lexical_master" if LEXICAL_SOURCE_MODE == "master_derived" else ANALYSIS_DATASET
COLLOC_DATA_DIR = colloc_root / "data" / COLLOC_DATA_KEY
FULL_TABLE = COLLOC_DATA_DIR / "collostructional_full.csv"
TOP_TABLE = COLLOC_DATA_DIR / "collostructional_top.csv"
SELECTIONS_JSON = COLLOC_DATA_DIR / "lexical_selections.json"
SELECTIONS_CSV = COLLOC_DATA_DIR / "lexical_selections_long.csv"

for path in [
    ANALYSIS_DATA_DIR,
    COLLOC_DATA_DIR,
    HEATMAP_PLOT_DIR,
    TRAJECTORY_PLOT_DIR,
    BUBBLE_PLOT_DIR,
    TIMELINE_PLOT_DIR,
    LEXICAL_CORPUS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("BASE_CORPUS_DIR:", BASE_CORPUS_DIR, "exists:", BASE_CORPUS_DIR.exists())
print("CLASSIFIED_CORPUS_PATH:", CLASSIFIED_CORPUS_PATH, "exists:", CLASSIFIED_CORPUS_PATH.exists())
print("MODEL_DIR:", MODEL_DIR, "exists:", MODEL_DIR.exists())
print("THRESHOLDS_PATH:", THRESHOLDS_PATH, "exists:", THRESHOLDS_PATH.exists())
print("ANALYSIS_DATA_DIR:", ANALYSIS_DATA_DIR)
print("LEXICAL_CORPUS_DIR:", LEXICAL_CORPUS_DIR)


## Build or load the lexical source and collostruction tables
---


In [ ]:
bigram_blocklist = {"dark_matter", "omega_omega"}
COLLO_BLOCKLIST = {"findings", "sample", "compare", "production"}
TFIDF_LISTS_DIR = SHARED_ASSETS_DIR / "lists" / "tfidf"
SKLEARN_STOPWORDS_PATH = TFIDF_LISTS_DIR / "sklearn_english_stopwords.txt"
FINAL_STOPWORDS_PATH = TFIDF_LISTS_DIR / "final_stopwords.txt"

def load_wordlist(path: Path) -> set[str]:
    with open(path, encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip() and not line.lstrip().startswith("#")}

STOPWORDS = load_wordlist(SKLEARN_STOPWORDS_PATH) | load_wordlist(FINAL_STOPWORDS_PATH)
def save_jsonl(rows, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def file_sha256(path: Path) -> str:
    import hashlib

    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_jsonl(path: Path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def bin_start_year(label: str) -> int:
    return int(str(label).split("-")[0])


def compact_bin_label(label: str) -> str:
    start, end = label.split("-")
    return f"{start}-{end[-2:]}"


def assign_time_bin(year: int) -> str | None:
    for label in BIN_LABELS_SOURCE:
        start, end = map(int, label.split("-"))
        if start <= year <= end:
            return label
    return None


def is_substantive(term: str) -> bool:
    tokens = term.replace("_", " ").split()
    if all(re.match(r"^[\d\.\-\+]+$", t) for t in tokens):
        return False
    if all(re.match(r"^\d{2,4}$", t) for t in tokens):
        return False
    return True


def is_valid_bigram(term: str) -> bool:
    if "_" not in term:
        return True
    weak = {"given", "include", "discuss", "present", "show", "found", "used", "based", "provide", "consider", "address", "suggest"}
    return not any(part in weak for part in term.split("_"))


def is_valid_term(term: str) -> bool:
    if not is_substantive(term):
        return False
    if term in bigram_blocklist or term in COLLO_BLOCKLIST:
        return False
    if not is_valid_bigram(term):
        return False
    return True


def filter_redundant(words: list[str]) -> list[str]:
    bigram_parts = set()
    for word in words:
        if "_" in word:
            bigram_parts.update(word.split("_"))
    return [word for word in words if "_" in word or word not in bigram_parts]


def tokenize(text: str) -> list[str]:
    if not text:
        return []
    tokens = tokenize_preproc_text(text)
    tokens = [tok for tok in tokens if tok not in STOPWORDS and len(tok) > 2]
    bigrams = [
        f"{tokens[i]}_{tokens[i + 1]}"
        for i in range(len(tokens) - 1)
        if tokens[i] not in STOPWORDS and tokens[i + 1] not in STOPWORDS
    ]
    return tokens + bigrams


def get_context_tokens(sent: str, start: int, end: int, window: int = WINDOW_CHARS) -> list[str]:
    left = max(0, start - window)
    right = min(len(sent), end + window)
    return tokenize(sent[left:start] + " " + sent[end:right])


def top_g2_terms_filtered(grp: pd.DataFrame, n: int = CELL_TOP_N) -> list[str]:
    grp = grp.sort_values("g2", ascending=False)
    selected = []
    for _, row in grp.iterrows():
        term = row["word"]
        if not is_valid_term(term):
            continue
        selected.append(term)
        if len(selected) >= n * 5:
            break
    return filter_redundant(selected)[:n]


class MentionDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
        }


class SciBERTClassifier(nn.Module):
    def __init__(self, base_model, n_labels=3, dropout=0.1):
        super().__init__()
        self.bert = base_model
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

def summarise_terms(cat: str, min_bins: int, max_rank: int | None = None) -> pd.DataFrame:
    cat_df = df_full[(df_full["category"] == cat) & (df_full["g2"] > g2_threshold)].copy()
    if cat_df.empty:
        return pd.DataFrame(columns=["mean_g2", "max_g2", "mean_pmi", "n_bins", "median_rank", "peak_bin", "peak_start"])

    cat_df["rank"] = cat_df.groupby("bin")["g2"].rank(ascending=False, method="first")
    rank_df = cat_df if max_rank is None else cat_df[cat_df["rank"] <= max_rank].copy()

    eligible = rank_df.groupby("word")["bin"].nunique()
    eligible = eligible[eligible >= min_bins].index
    subset = cat_df[cat_df["word"].isin(eligible)].copy()
    if subset.empty:
        return pd.DataFrame(columns=["mean_g2", "max_g2", "mean_pmi", "n_bins", "median_rank", "peak_bin", "peak_start"])

    summary = subset.groupby("word").agg(
        mean_g2=("g2", "mean"),
        max_g2=("g2", "max"),
        mean_pmi=("pmi", "mean"),
        n_bins=("bin", "nunique"),
        median_rank=("rank", "median"),
        freq_corpus=("freq_corpus", "max"),
    )
    peak_rows = subset.loc[subset.groupby("word")["g2"].idxmax(), ["word", "bin"]].rename(columns={"bin": "peak_bin"})
    summary = summary.join(peak_rows.set_index("word"))
    summary["peak_start"] = summary["peak_bin"].map(bin_start_year)
    return summary


def select_terms_from_summary(summary: pd.DataFrame, top_k: int, sort_cols: list[str], ascending: list[bool]) -> tuple[list[str], pd.DataFrame]:
    if summary.empty:
        return [], summary
    ranked = summary.sort_values(sort_cols, ascending=ascending)
    terms = filter_redundant(ranked.index.tolist())[:top_k]
    return terms, ranked.loc[terms].copy()


def select_terms_for_heatmap(cat: str) -> tuple[list[str], pd.DataFrame]:
    summary = summarise_terms(cat, min_bins=HEATMAP_MIN_BINS, max_rank=HEATMAP_MAX_RANK)
    return select_terms_from_summary(summary, HEATMAP_N, ["mean_g2", "max_g2", "n_bins", "peak_start"], [False, False, False, True])


def select_terms_for_trajectory(cat: str) -> tuple[list[str], pd.DataFrame]:
    summary = summarise_terms(cat, min_bins=TRAJECTORY_MIN_BINS, max_rank=TRAJECTORY_MAX_RANK)
    return select_terms_from_summary(summary, TRAJECTORY_TOP_K, ["median_rank", "mean_g2", "n_bins", "peak_start"], [True, False, False, True])


def select_terms_for_bubble(cat: str) -> tuple[list[str], pd.DataFrame]:
    summary = summarise_terms(cat, min_bins=BUBBLE_MIN_BINS, max_rank=BUBBLE_MAX_RANK)
    return select_terms_from_summary(summary, BUBBLE_TOP_K, ["median_rank", "mean_g2", "n_bins", "peak_start"], [True, False, False, True])


def select_terms_for_timeline(cat: str) -> tuple[list[str], pd.DataFrame]:
    summary = summarise_terms(cat, min_bins=TIMELINE_MIN_BINS, max_rank=TIMELINE_MAX_RANK)
    return select_terms_from_summary(summary, TIMELINE_TOP_K, ["median_rank", "mean_g2", "n_bins", "peak_start"], [True, False, False, True])

def build_derived_corpus_manifest(run_id: str, output_dir: Path, written_bins: list[str], final_bin_counts: dict[str, int], bin_content_hashes: dict[str, str], source_records_path: Path, source_manifest: dict, master_manifest: dict, base_run_id: str) -> dict:
    return {
        "run_id": run_id,
        "source_parquet": master_manifest.get("source_parquet"),
        "derived_from_master_run_id": MASTER_RUN_ID,
        "derived_from_lexical_source": str(source_records_path),
        "base_run_id": base_run_id,
        "output_dir": str(output_dir),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "written_bins": written_bins,
        "n_written_bins": len(written_bins),
        "final_bin_counts": final_bin_counts,
        "final_total_examples": int(sum(final_bin_counts.values())),
        "bin_content_hashes": bin_content_hashes,
        "filters": master_manifest.get("filters", {}),
        "sampling": {
            "type": "uncapped_lexical_master",
            "base_run_id": base_run_id,
            "classified_source": str(source_records_path),
            "bin_labels": source_manifest.get("bin_labels", []),
        },
    }


def assign_labels(prob_A, prob_B, prob_C):
    probs = {"A": prob_A, "B": prob_B, "C": prob_C}
    return [label for label in LABEL_SPACE if probs[label] >= LABEL_THRESHOLDS[label]]


In [ ]:
with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    threshold_info = json.load(f)
LABEL_THRESHOLDS = {label: float(threshold_info["per_label"][label]["threshold"]) for label in LABEL_SPACE}
print("Using lexical classification thresholds:", LABEL_THRESHOLDS)
print("Building lexical source from:", LEXICAL_SOURCE_MODE_LABELS[LEXICAL_SOURCE_MODE])

if LEXICAL_MENTIONS_FILE.exists() and not FORCE_REBUILD_LEXICAL_SOURCE:
    lexical_mentions = load_jsonl(LEXICAL_MENTIONS_FILE)
    print(f"Loaded lexical mentions ({LEXICAL_SOURCE_MODE}) -> {LEXICAL_MENTIONS_FILE}")
else:
    lexical_mentions = []
    if LEXICAL_SOURCE_MODE == "master_derived":
        master_df = pd.read_parquet(MASTER_PARQUET)
        required_cols = {"example_key", "year", "time_bin_current", "bibcode", "sent", "start", "end", "has_math", "source_row_index", "mention_order"}
        missing_cols = required_cols - set(master_df.columns)
        if missing_cols:
            raise ValueError(f"Master mention table missing required columns: {sorted(missing_cols)}")

        master_df = master_df[master_df["time_bin_current"].notna()].copy()
        master_df = master_df[master_df["time_bin_current"].isin(BIN_LABELS_SOURCE)].copy()
        master_df = master_df.sort_values(["source_row_index", "mention_order", "bibcode", "start", "end"], kind="mergesort")
        print(f"Rows entering master-derived lexical source: {len(master_df):,}")

        for row in master_df[["example_key", "time_bin_current", "year", "bibcode", "sent", "start", "end", "has_math"]].to_dict(orient="records"):
            lexical_mentions.append({
                "example_key": row["example_key"],
                "bin": str(row["time_bin_current"]),
                "year": int(row["year"]),
                "bibcode": str(row["bibcode"]),
                "sent": " ".join(str(row["sent"]).split()),
                "start": int(row["start"]),
                "end": int(row["end"]),
                "has_math": bool(row.get("has_math", False)),
            })

        source_manifest = {
            "dataset": ANALYSIS_DATASET,
            "lexical_source_mode": LEXICAL_SOURCE_MODE,
            "lexical_source_label": LEXICAL_SOURCE_MODE_LABELS[LEXICAL_SOURCE_MODE],
            "derived_from_master_run_id": MASTER_RUN_ID,
            "master_parquet": str(MASTER_PARQUET),
            "master_manifest": str(MASTER_MANIFEST),
            "base_run_id": BASE_RUN_ID,
            "bin_labels": BIN_LABELS_SOURCE,
            "n_mentions": len(lexical_mentions),
            "master_total_mentions": int(master_manifest["row_counts"]["mentions_written"] if "row_counts" in master_manifest and "mentions_written" in master_manifest["row_counts"] else len(master_df)),
            "counts_by_bin": pd.Series([row["bin"] for row in lexical_mentions]).value_counts().sort_index().to_dict(),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
    else:
        if not BASE_CORPUS_MANIFEST.exists():
            raise FileNotFoundError(f"Base corpus manifest not found: {BASE_CORPUS_MANIFEST}")
        with open(BASE_CORPUS_MANIFEST, "r", encoding="utf-8") as f:
            base_corpus_manifest = json.load(f)

        rows_entering = 0
        for bin_label in BIN_LABELS_SOURCE:
            in_path = BASE_CORPUS_DIR / f"{bin_label}.jsonl"
            if not in_path.exists():
                continue
            rows = load_jsonl(in_path)
            rows_entering += len(rows)
            for row in rows:
                lexical_mentions.append({
                    "example_key": row.get("example_key"),
                    "bin": str(row.get("time_bin", bin_label)),
                    "year": int(row["year"]),
                    "bibcode": str(row["bibcode"]),
                    "sent": " ".join(str(row["sent"]).split()),
                    "start": int(row["start"]),
                    "end": int(row["end"]),
                    "has_math": bool(row.get("has_math", False)),
                })
        lexical_mentions.sort(key=lambda row: (row["bin"], row["year"], row.get("bibcode") or "", row["start"], row["end"]))
        print(f"Rows entering analysis-corpus lexical source: {rows_entering:,}")

        source_manifest = {
            "dataset": ANALYSIS_DATASET,
            "lexical_source_mode": LEXICAL_SOURCE_MODE,
            "lexical_source_label": LEXICAL_SOURCE_MODE_LABELS[LEXICAL_SOURCE_MODE],
            "derived_from_base_run_id": BASE_RUN_ID,
            "base_corpus_root": str(BASE_CORPUS_ROOT),
            "base_corpus_manifest": str(BASE_CORPUS_MANIFEST),
            "bin_labels": BIN_LABELS_SOURCE,
            "n_mentions": len(lexical_mentions),
            "base_total_mentions": int(sum(base_corpus_manifest.get("final_bin_counts", {}).values())),
            "counts_by_bin": pd.Series([row["bin"] for row in lexical_mentions]).value_counts().sort_index().to_dict(),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }

    save_jsonl(lexical_mentions, LEXICAL_MENTIONS_FILE)
    with open(LEXICAL_SOURCE_MANIFEST, "w", encoding="utf-8") as f:
        json.dump(source_manifest, f, indent=2)
    print(f"Saved lexical mentions ({LEXICAL_SOURCE_MODE}) -> {LEXICAL_MENTIONS_FILE}")
    print(f"Saved source manifest -> {LEXICAL_SOURCE_MANIFEST}")

if LEXICAL_CLASSIFIED_FILE.exists() and not FORCE_REBUILD_LEXICAL_CLASSIFIED:
    records = load_jsonl(LEXICAL_CLASSIFIED_FILE)
    print(f"Loaded lexical classified corpus ({LEXICAL_SOURCE_MODE}) -> {LEXICAL_CLASSIFIED_FILE}")
elif LEXICAL_SOURCE_MODE == "analysis_corpus" and CLASSIFIED_CORPUS_PATH.exists():
    records = load_jsonl(CLASSIFIED_CORPUS_PATH)
    records = [
        {
            "id": row.get("id"),
            "example_key": row.get("example_key"),
            "bin": row["bin"],
            "year": int(row["year"]),
            "bibcode": row.get("bibcode"),
            "sent": " ".join(str(row["sent"]).split()),
            "start": int(row["start"]),
            "end": int(row["end"]),
            "has_math": bool(row.get("has_math", False)),
            "prob_A": float(row["prob_A"]),
            "prob_B": float(row["prob_B"]),
            "prob_C": float(row["prob_C"]),
            "labels": list(row.get("labels", [])),
        }
        for row in records
        if row.get("bin") in BIN_LABELS_SOURCE
    ]
    records.sort(key=lambda row: (row["bin"], row["year"], row.get("bibcode") or "", row["start"], row["end"]))
    save_jsonl(records, LEXICAL_CLASSIFIED_FILE)
    print(f"Reused existing 02c classifications from -> {CLASSIFIED_CORPUS_PATH}")
    print(f"Saved lexical classified corpus ({LEXICAL_SOURCE_MODE}) -> {LEXICAL_CLASSIFIED_FILE}")
else:
    tokenizer = AutoTokenizer.from_pretrained(SCIBERT_BASE, use_fast=True)
    model = SciBERTClassifier(AutoModel.from_pretrained(SCIBERT_BASE).to(DEVICE), n_labels=3).to(DEVICE)
    model.load_state_dict(torch.load(BEST_MODEL, map_location=DEVICE, weights_only=True))
    model.eval()

    dataset = MentionDataset([row["sent"] for row in lexical_mentions], tokenizer)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    all_probs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Classify lexical corpus ({LEXICAL_SOURCE_MODE})"):
            logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
            all_probs.append(torch.sigmoid(logits).cpu().numpy())

    records = []
    for i, (row, prob) in enumerate(zip(lexical_mentions, np.vstack(all_probs))):
        prob_A, prob_B, prob_C = map(float, prob)
        records.append({
            "id": f"{row['bin']}_{i}",
            "example_key": row.get("example_key"),
            "bin": row["bin"],
            "year": int(row["year"]),
            "bibcode": row.get("bibcode"),
            "sent": row["sent"],
            "start": int(row["start"]),
            "end": int(row["end"]),
            "has_math": bool(row.get("has_math", False)),
            "prob_A": prob_A,
            "prob_B": prob_B,
            "prob_C": prob_C,
            "labels": assign_labels(prob_A, prob_B, prob_C),
        })

    save_jsonl(records, LEXICAL_CLASSIFIED_FILE)
    print(f"Saved lexical classified corpus ({LEXICAL_SOURCE_MODE}) -> {LEXICAL_CLASSIFIED_FILE}")

print(f"Loaded {len(records):,} classified mentions ({LEXICAL_SOURCE_MODE}) across {len(BIN_LABELS_SOURCE)} bins")
print(pd.Series([row['bin'] for row in records]).value_counts().sort_index())

if FULL_TABLE.exists() and TOP_TABLE.exists() and not FORCE_REBUILD_COLLOC:
    df_full = pd.read_csv(FULL_TABLE)
    df_top = pd.read_csv(TOP_TABLE)
    print(f"Loaded cached full table -> {FULL_TABLE}")
    print(f"Loaded cached top table -> {TOP_TABLE}")
else:
    print(f"Building collostructional statistics from lexical corpus ({LEXICAL_SOURCE_MODE})...")
    corpus_freq = Counter()
    cell_freq = defaultdict(lambda: defaultdict(Counter))
    cell_n = defaultdict(lambda: defaultdict(float))

    for row in records:
        tokens = get_context_tokens(row["sent"], int(row["start"]), int(row["end"]))
        if not tokens:
            continue
        bin_label = row["bin"]
        corpus_freq.update(tokens)
        for cat in CATEGORIES:
            weight = float(row.get(f"prob_{cat}", 0.0))
            if weight <= CELL_WEIGHT_MIN_PROB:
                continue
            for token in tokens:
                cell_freq[bin_label][cat][token] += weight
            cell_n[bin_label][cat] += weight

    vocab = {word for word, count in corpus_freq.items() if count >= MIN_FREQ and is_valid_term(word)}
    corpus_total = float(sum(corpus_freq.values()))
    words = list(vocab)
    print(f"Vocabulary size (after filtering): {len(vocab):,}")

    results = []
    for bin_label in BIN_LABELS_SOURCE:
        for cat in CATEGORIES:
            if cell_n[bin_label][cat] == 0:
                continue

            cell_total = float(sum(cell_freq[bin_label][cat].values()))
            O11 = np.array([cell_freq[bin_label][cat].get(word, 0.0) for word in words], dtype=float)
            O12 = np.array([corpus_freq[word] - cell_freq[bin_label][cat].get(word, 0.0) for word in words], dtype=float)
            O21 = cell_total - O11
            O22 = corpus_total - cell_total - O12
            N = O11 + O12 + O21 + O22

            E11 = (O11 + O12) * (O11 + O21) / N
            E12 = (O11 + O12) * (O12 + O22) / N
            E21 = (O21 + O22) * (O11 + O21) / N
            E22 = (O21 + O22) * (O12 + O22) / N

            def safe_log(o, e):
                with np.errstate(divide="ignore", invalid="ignore"):
                    return np.where((o > 0) & (e > 0), o * np.log(o / e), 0.0)

            g2 = 2 * (safe_log(O11, E11) + safe_log(O12, E12) + safe_log(O21, E21) + safe_log(O22, E22))
            sign = np.where(
                O11 / np.maximum(O11 + O21, 1) > O12 / np.maximum(O12 + O22, 1),
                1,
                -1,
            )
            g2 *= sign

            pmi_vals = np.where(
                O11 > 0,
                np.log2(np.maximum(O11 * N / (np.maximum(O11 + O12, 1) * np.maximum(O11 + O21, 1)), 1e-10)),
                -np.inf,
            )

            for idx, word in enumerate(words):
                results.append({
                    "bin": bin_label,
                    "category": cat,
                    "word": word,
                    "freq_cell": O11[idx],
                    "freq_corpus": corpus_freq[word],
                    "n_mentions": cell_n[bin_label][cat],
                    "g2": float(g2[idx]),
                    "pmi": float(pmi_vals[idx]),
                })

            print(f"  {bin_label} {cat} done")

    df_full = pd.DataFrame(results)
    df_full.to_csv(FULL_TABLE, index=False)
    print(f"Full table: {len(df_full):,} rows -> {FULL_TABLE}")

    top_rows = []
    for (bin_label, cat), grp in df_full.groupby(["bin", "category"]):
        top_words = top_g2_terms_filtered(grp, n=CELL_TOP_N)
        top = grp[grp["word"].isin(top_words)].copy()
        top["rank"] = top["word"].map({word: i for i, word in enumerate(top_words, start=1)})
        top = top.sort_values("rank")
        top_rows.append(top)

    df_top = pd.concat(top_rows, ignore_index=True)
    df_top.to_csv(TOP_TABLE, index=False)
    print(f"Top-{CELL_TOP_N} table: {len(df_top):,} rows -> {TOP_TABLE}")

BINS = sorted(df_full["bin"].unique(), key=bin_start_year)
BIN_LABELS = [compact_bin_label(b).replace("-", "–", 1) for b in BINS]
print(df_full.shape, df_top.shape)
print(BINS)


## Export lexical master as a corpus namespace

This section converts the classified lexical source into a standard corpus directory-

---


In [ ]:
if LEXICAL_CORPUS_MANIFEST.exists() and not FORCE_REBUILD_LEXICAL_CORPUS:
    with open(LEXICAL_CORPUS_MANIFEST, "r", encoding="utf-8") as f:
        lexical_corpus_manifest = json.load(f)
    print(f"Loaded lexical corpus manifest -> {LEXICAL_CORPUS_MANIFEST}")
else:
    for old_path in sorted(LEXICAL_CORPUS_DIR.glob("*.jsonl")):
        old_path.unlink()

    records_df = pd.DataFrame(records).copy()
    records_df = records_df.sort_values(["bin", "year", "bibcode", "start", "end"], kind="mergesort")

    written_bins = []
    final_bin_counts = {}
    bin_content_hashes = {}

    for bin_label in BIN_LABELS_SOURCE:
        sub = records_df[records_df["bin"] == bin_label].copy()
        if sub.empty:
            continue

        rows = []
        for row in sub.to_dict(orient="records"):
            row_out = dict(row)
            row_out["time_bin"] = row_out["bin"]
            rows.append(row_out)

        out_path = LEXICAL_CORPUS_DIR / f"{bin_label}.jsonl"
        save_jsonl(rows, out_path)
        written_bins.append(bin_label)
        final_bin_counts[bin_label] = len(rows)
        bin_content_hashes[bin_label] = file_sha256(out_path)
        print(f"{bin_label}: wrote {len(rows):,} rows -> {out_path}")

    with open(LEXICAL_SOURCE_MANIFEST, "r", encoding="utf-8") as f:
        lexical_source_manifest = json.load(f)

    lexical_corpus_manifest = build_derived_corpus_manifest(
        run_id=LEXICAL_CORPUS_RUN_ID,
        output_dir=LEXICAL_CORPUS_DIR,
        written_bins=written_bins,
        final_bin_counts=final_bin_counts,
        bin_content_hashes=bin_content_hashes,
        source_records_path=LEXICAL_CLASSIFIED_FILE,
        source_manifest=lexical_source_manifest,
        master_manifest=master_manifest,
        base_run_id=BASE_RUN_ID,
    )

    with open(LEXICAL_CORPUS_MANIFEST, "w", encoding="utf-8") as f:
        json.dump(lexical_corpus_manifest, f, indent=2)
    print(f"Saved lexical corpus manifest -> {LEXICAL_CORPUS_MANIFEST}")

print("Lexical corpus counts:")
display(pd.Series(lexical_corpus_manifest["final_bin_counts"]).sort_index().to_frame("count"))


## Derive lexical selections dynamically from the full collostruction table
---


In [ ]:
CHI_OPTIONS = {
    "loose": 3.84,
    "moderate": 6.63,
    "strict": 10.83,
    "very_strict": 15.13,
}
CHI_OPTION = "very_strict"
g2_threshold = CHI_OPTIONS[CHI_OPTION]

HEATMAP_N = 15
HEATMAP_MIN_BINS = 2
HEATMAP_MAX_RANK = 25

TRAJECTORY_TOP_K = 10
TRAJECTORY_MIN_BINS = 2
TRAJECTORY_MAX_RANK = 25

BUBBLE_TOP_K = 10
BUBBLE_MIN_BINS = 2
BUBBLE_MAX_RANK = 20

TIMELINE_TOP_K = 20
TIMELINE_MIN_BINS = 2
TIMELINE_MAX_RANK = 25

In [ ]:
# EXPLORATORY

CHI_OPTION = "moderate"
HEATMAP_N = 20
HEATMAP_MIN_BINS = 2
HEATMAP_MAX_RANK = 30

TRAJECTORY_TOP_K = 15
TRAJECTORY_MIN_BINS = 2
TRAJECTORY_MAX_RANK = 30

BUBBLE_TOP_K = 15
BUBBLE_MIN_BINS = 2
BUBBLE_MAX_RANK = 25

TIMELINE_TOP_K = 25
TIMELINE_MIN_BINS = 2
TIMELINE_MAX_RANK = 30


In [ ]:
# PAPER [CONSERVATIVE]

CHI_OPTION = "strict"
HEATMAP_N = 12
HEATMAP_MIN_BINS = 2
HEATMAP_MAX_RANK = 20

TRAJECTORY_TOP_K = 8
TRAJECTORY_MIN_BINS = 2
TRAJECTORY_MAX_RANK = 25

BUBBLE_TOP_K = 8
BUBBLE_MIN_BINS = 2
BUBBLE_MAX_RANK = 25

TIMELINE_TOP_K = 12
TIMELINE_MIN_BINS = 2
TIMELINE_MAX_RANK = 25


In [ ]:
selection_specs = {
    "heatmap": select_terms_for_heatmap,
    "trajectory": select_terms_for_trajectory,
    "bubble": select_terms_for_bubble,
    "timeline": select_terms_for_timeline,
}

selected_terms: dict[str, dict[str, list[str]]] = {cat: {} for cat in CATEGORIES}
selection_rows = []

for cat in CATEGORIES:
    print(f"\nCategory {cat} ({CAT_LABELS[cat].strip()}):")
    for kind, selector in selection_specs.items():
        terms, summary = selector(cat)
        selected_terms[cat][kind] = terms
        print(f"  {kind}: {terms}")
        if not summary.empty:
            ordered = summary.reset_index().rename(columns={"index": "word"}).copy()
            ordered["category"] = cat
            ordered["selection_kind"] = kind
            ordered["display_rank"] = range(1, len(ordered) + 1)
            selection_rows.append(ordered)

selection_long = pd.concat(selection_rows, ignore_index=True) if selection_rows else pd.DataFrame()
selection_long.to_csv(SELECTIONS_CSV, index=False)
with open(SELECTIONS_JSON, "w", encoding="utf-8") as f:
    json.dump(selected_terms, f, ensure_ascii=False, indent=2)

print(f"Saved lexical selections -> {SELECTIONS_JSON}")
print(f"Saved lexical selection summary -> {SELECTIONS_CSV}")


## Heatmaps
---


In [ ]:
import matplotlib as mpl
mpl.rcParams["font.family"] = "Stix Two Math"

def build_heatmap_matrix(cat: str, words: list[str]) -> np.ndarray:
    cat_df = df_full[df_full["category"] == cat]
    matrix = np.zeros((len(words), len(BINS)))
    for j, bin_label in enumerate(BINS):
        sub = cat_df[cat_df["bin"] == bin_label].set_index("word")
        for i, word in enumerate(words):
            if word in sub.index:
                matrix[i, j] = sub.loc[word, "g2"]
    return matrix

def zscore_rows(matrix: np.ndarray) -> np.ndarray:
    row_mean = matrix.mean(axis=1, keepdims=True)
    row_std = matrix.std(axis=1, keepdims=True) + 1e-10
    return (matrix - row_mean) / row_std

all_heatmap_z = []
for cat in CATEGORIES:
    words = selected_terms[cat]["heatmap"]
    if words:
        all_heatmap_z.append(zscore_rows(build_heatmap_matrix(cat, words)).ravel())

VMIN = np.percentile(np.concatenate(all_heatmap_z), 2) if all_heatmap_z else -2.0
VMAX = np.percentile(np.concatenate(all_heatmap_z), 98) if all_heatmap_z else 2.0

CATEGORY_HEATMAP_CMAPS = {
    "A": sns.cubehelix_palette(start=0.15, light=1.0, dark=0.38, rot=-0.45, as_cmap=True),
    "B": sns.cubehelix_palette(start=2.75, light=1.0, dark=0.38, rot=-0.45, as_cmap=True),
}

def plot_heatmap(cat: str, ax=None, save: bool = False, show: bool = True):
    words = selected_terms[cat]["heatmap"]
    word_labels = [w.replace("_", " ") for w in words]
    if not words:
        raise ValueError(f"No heatmap words available for category {cat}")

    matrix_z = zscore_rows(build_heatmap_matrix(cat, words))
    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, max(3, HEATMAP_N * 0.45)))
        created_fig = True
    else:
        fig = ax.figure

    fig.patch.set_facecolor(theme["bg"])
    ax.set_facecolor(theme["bg"])
    ax.grid(False)

    cmap = sns.cubehelix_palette(start=0, light=1.0, dark=0.4, rot=-.5, as_cmap=True)
    im = ax.pcolormesh(matrix_z, cmap=cmap, vmin=VMIN, vmax=VMAX, linewidth=1.0, edgecolors=theme["bg"])
    ax.set_xticks(np.arange(len(BINS)) + 0.5)
    ax.set_xticklabels(BIN_LABELS, color=theme["fg"], fontsize=9)
    ax.set_yticks(np.arange(len(words)) + 0.5)
    ax.set_yticklabels(word_labels, color=theme["fg"], fontsize=8.5)
    ax.invert_yaxis()

    for spine in ax.spines.values():
        spine.set_edgecolor(theme["bg"])

    ax.set_title(
        f"Collocate association strength over time {CAT_LABELS[cat]}",
        color=theme["fg"],
        fontsize=11,
        pad=10,
        loc="left",
    )

    cbar = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.05, pad=0.12)
    cbar.ax.tick_params(labelsize=8, color=theme["fg"], width=0.5)
    cbar.ax.set_title("G² (z-scored per term)", color=theme["fg"], fontsize=9, pad=6)
    cbar.outline.set_edgecolor(theme["fg"])
    cbar.outline.set_linewidth(0.5)
    plt.setp(cbar.ax.get_xticklabels(), color=theme["fg"], fontsize=8)

    if save and created_fig:
        out = HEATMAP_PLOT_DIR / f"heatmap_{cat}.pdf"
        plt.savefig(out, dpi=300, bbox_inches="tight", facecolor=theme["bg"])
        record_figure_output(out)
        print(f"Saved -> {out}")

    if show and created_fig:
        plt.show()


for cat in CATEGORIES:
    plot_heatmap(cat, save=True)

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(11, 8), facecolor=theme["bg"])
for ax, cat in zip(axes, CATEGORIES):
    words = selected_terms[cat]["heatmap"]
    matrix_z = zscore_rows(build_heatmap_matrix(cat, words))
    im = ax.pcolormesh(matrix_z, cmap=CATEGORY_HEATMAP_CMAPS[cat], vmin=VMIN, vmax=VMAX, linewidth=1.0, edgecolors=theme["bg"])
    ax.set_facecolor(theme["bg"])
    ax.grid(False)
    ax.set_yticks(np.arange(len(words)) + 0.5)
    ax.set_yticklabels(words, color=theme["fg"], fontsize=8.5)
    ax.invert_yaxis()
    ax.set_title(CAT_LABELS[cat], color=theme["fg"], fontsize=10, loc="left", pad=6)
    if ax is axes[-1]:
        ax.set_xticks(np.arange(len(BINS)) + 0.5)
        ax.set_xticklabels(BIN_LABELS, color=theme["fg"], fontsize=8, rotation=45, ha="right")
    else:
        ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

cbar = fig.colorbar(im, ax=axes, orientation="vertical", fraction=0.015, pad=0.02)
cbar.ax.set_title(r"$G^2\, z$", color=theme["fg"], fontsize=8, pad=6)
cbar.ax.tick_params(labelsize=7, color=theme["fg"], width=0.5)
plt.setp(cbar.ax.get_yticklabels(), color=theme["fg"], fontsize=7)
cbar.outline.set_edgecolor(theme["fg"])

fig.suptitle("Collostruction heatmaps by category", color=theme["fg"], fontsize=13, y=0.99)
#plt.tight_layout(rect=[0, 0, 0.97, 0.98])
heatmap_grid_out = HEATMAP_PLOT_DIR / "heatmap_grid.png"
plt.savefig(heatmap_grid_out, dpi=300, bbox_inches="tight", facecolor=theme["bg"])
record_figure_output(heatmap_grid_out)
plt.show()
print(f"Saved -> {heatmap_grid_out}")


## PMI trajectories
---


In [ ]:
def plot_trajectory(cat: str, ax=None, save: bool = False, show: bool = True, show_legend: bool = True):
    terms = selected_terms[cat]["trajectory"]
    if not terms:
        raise ValueError(f"No trajectory terms available for category {cat}")

    cat_df = df_full[df_full["category"] == cat]
    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(13, 5))
        created_fig = True
    else:
        fig = ax.figure

    fig.patch.set_facecolor(theme["bg"])
    ax.set_facecolor(theme["bg"])

    #cmap = sns.color_palette("hls", 8, as_cmap=True)
    cmap = sns.color_palette("tab10", as_cmap=True)
    colors_by_term = {term: mcolors.to_hex(cmap(i / max(len(terms) - 1, 1))) for i, term in enumerate(terms)}
    x = list(range(len(BINS)))

    for term in terms:
        pmi_vals = []
        for bin_label in BINS:
            row = cat_df[(cat_df["bin"] == bin_label) & (cat_df["word"] == term)]
            pmi_vals.append(row["pmi"].iloc[0] if len(row) else np.nan)

        pmi_arr = np.array(pmi_vals, dtype=float)
        if np.sum(~np.isnan(pmi_arr)) < 3:
            continue

        linestyle = "-" if "_" in term else "-"
        ax.plot(
            x,
            pmi_arr,
            marker="o",
            linewidth=1.25,
            markersize=7,
            markeredgecolor=theme["bg"],
            markeredgewidth=0.75,
            color=colors_by_term[term],
            label=term.replace("_", " "),
            linestyle=linestyle,
            alpha=1.0,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(BIN_LABELS, color=theme["fg"], fontsize=9)
    ax.tick_params(colors=theme["fg"], length=0)
    ax.set_ylabel("PMI", color=theme["fg"], fontsize=9)
    ax.grid(alpha=0.2, color=theme["fg"])

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(f"Collocate PMI trajectories {CAT_LABELS[cat]}", color=theme["fg"], fontsize=11, pad=10, loc="left")

    if show_legend:
        leg = ax.legend(
            ncol=1,
            fontsize=9,
            facecolor=theme["bg"],
            edgecolor="none",
            labelcolor=theme["fg"],
            title="Associated terms",
            alignment="left",
            loc="upper left",
            title_fontsize=10,
            bbox_to_anchor=(1.01, 1.0),
            frameon=False,
        )
        leg.get_title().set_color(theme["fg"])

    if save and created_fig:
        out = TRAJECTORY_PLOT_DIR / f"trajectory_{cat}.png"
        plt.savefig(out, dpi=300, bbox_inches="tight", facecolor=theme["bg"])
        record_figure_output(out)
        print(f"Saved -> {out}")

    if show and created_fig:
        plt.show()


for cat in CATEGORIES:
    plot_trajectory(cat, save=True)

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(12, 12), sharex=True, facecolor=theme["bg"])
for ax, cat in zip(axes, CATEGORIES):
    plot_trajectory(cat, ax=ax, show=False, show_legend=False)
axes[-1].set_xticks(range(len(BINS)))
axes[-1].set_xticklabels(BIN_LABELS, rotation=45, ha="right", color=theme["fg"])
fig.suptitle("Collocate PMI trajectories by category", color=theme["fg"], fontsize=14, y=0.985)
plt.tight_layout(rect=[0, 0, 1, 0.98])
trajectory_grid_out = TRAJECTORY_PLOT_DIR / "trajectory_grid.png"
plt.savefig(trajectory_grid_out, dpi=300, bbox_inches="tight", facecolor=theme["bg"])
record_figure_output(trajectory_grid_out)
plt.show()
print(f"Saved -> {trajectory_grid_out}")


## Bubble grid plot
---


In [ ]:
def plot_bubble_grid():
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"wspace": 0.35}, facecolor=theme["bg"])

    for ax, cat in zip(axes, CATEGORIES):
        words = selected_terms[cat]["bubble"]
        word_labels = [w.replace("_", " ") for w in words]
        cat_df = df_full[(df_full["category"] == cat) & (df_full["word"].isin(words)) & (df_full["g2"] > g2_threshold)].copy()
        if cat_df.empty:
            ax.axis("off")
            continue

        cat_df["x"] = cat_df["bin"].map({b: i for i, b in enumerate(BINS)})
        word_order = {word: i for i, word in enumerate(words)}
        cat_df["y"] = cat_df["word"].map(word_order)

        #cmap = sns.blend_palette([theme["bg"], colors[cat]], as_cmap=True)
        cmap = sns.cubehelix_palette(start=0, light=1.0, dark=0.4, rot=-.5, as_cmap=True)
        size_scale = 600 / max(cat_df["g2"].max(), 1e-6)

        ax.scatter(
            cat_df["x"],
            cat_df["y"],
            s=cat_df["g2"] * size_scale,
            c=cat_df["g2"],
            cmap=cmap,
            alpha=0.9,
            edgecolors=theme["fg"],
            linewidths=0.25,
        )

        ax.set_xticks(range(len(BINS)))
        ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", color=theme["fg"], fontsize=8)
        ax.set_yticks(range(len(words)))
        ax.set_yticklabels(word_labels, color=theme["fg"], fontsize=8.5)
        ax.invert_yaxis()
        ax.set_facecolor(theme["bg"])
        ax.set_title(CAT_LABELS[cat], color=theme["fg"], fontsize=11, loc="left")
        ax.grid(axis="x", linestyle="--", alpha=0.2, color=theme["fg"])

        for spine in ax.spines.values():
            spine.set_visible(False)

    fig.suptitle("Bubble grid of collostruction strength by category", color=theme["fg"], fontsize=13, y=0.98)
    #plt.tight_layout(rect=[0, 0, 1, 0.96])
    out = BUBBLE_PLOT_DIR / "bubble_grid.png"
    plt.savefig(out, dpi=300, bbox_inches="tight", facecolor=theme["bg"])
    record_figure_output(out)
    plt.show()
    print(f"Saved -> {out}")


plot_bubble_grid()


## Presence timelines
---


In [ ]:
def build_presence_matrix(cat: str, words: list[str]) -> np.ndarray:
    cat_df = df_full[(df_full["category"] == cat) & (df_full["g2"] > g2_threshold)]
    word_to_idx = {word: i for i, word in enumerate(words)}
    presence = np.zeros((len(words), len(BINS)), dtype=int)
    for j, bin_label in enumerate(BINS):
        bin_words = set(cat_df[cat_df["bin"] == bin_label]["word"].tolist())
        for word in words:
            if word in bin_words:
                presence[word_to_idx[word], j] = 1
    return presence


def plot_collostruction_timeline(cat: str, words=None, save: bool = False, show: bool = True, ax=None):
    if words is None:
        words = selected_terms[cat]["timeline"]
    if not words:
        raise ValueError(f"No timeline words available for category {cat}")

    word_to_idx = {word: i for i, word in enumerate(words)}
    word_labels = [w.replace("_", " ") for w in words]
    presence = build_presence_matrix(cat, words)
    cmap = sns.color_palette("tab10", as_cmap=True)
    #cmap = CATEGORY_TRAJECTORY_CMAPS[cat]
    colors_by_word = {word: cmap(i / max(1, len(words) - 1)) for i, word in enumerate(words)}

    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(words) - 1.5)), facecolor=theme["bg"])
        created_fig = True
    else:
        fig = ax.figure

    ax.set_facecolor(theme["bg"])
    for word, i in word_to_idx.items():
        for j in range(len(BINS)):
            if presence[i, j]:
                ax.plot(
                    j,
                    i,
                    "o",
                    color=colors_by_word[word],
                    markersize=6,
                    markeredgewidth=0.0,
                    markeredgecolor=theme["fg"],
                )
                if j > 0 and presence[i, j - 1]:
                    ax.add_line(
                        mlines.Line2D(
                            [j - 1, j],
                            [i, i],
                            color=colors_by_word[word],
                            linewidth=1.15,
                            alpha=0.9,
                        )
                    )

    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(word_labels, fontsize=10, color=theme["fg"])
    ax.set_xticks(range(len(BINS)))
    ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", color=theme["fg"], fontsize=8)
    ax.invert_yaxis()
    ax.tick_params(length=0, colors=theme["fg"])
    ax.grid(axis="x", linestyle="--", alpha=0.25, color=theme["fg"])

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(f"Collostruction timeline\n{CAT_LABELS[cat]}", color=theme["fg"], fontsize=12, loc="left")

    if save and created_fig:
        out = TIMELINE_PLOT_DIR / f"timeline_{cat}.pdf"
        plt.savefig(out, dpi=300, bbox_inches="tight", facecolor=theme["bg"])
        record_figure_output(out)
        print(f"Saved -> {out}")

    if show and created_fig:
        plt.show()


for cat in CATEGORIES:
    plot_collostruction_timeline(cat, save=True)

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(12, 14), sharex=True, facecolor=theme["bg"])
for ax, cat in zip(axes, CATEGORIES):
    plot_collostruction_timeline(cat, ax=ax, show=False)
axes[-1].set_xticks(range(len(BINS)))
axes[-1].set_xticklabels(BIN_LABELS, rotation=45, ha="right", color=theme["fg"])
fig.suptitle("Collostruction timelines by category", color=theme["fg"], fontsize=14, y=0.98)
plt.tight_layout(rect=[0, 0.02, 1, 0.97])
out = TIMELINE_PLOT_DIR / "timeline_grid.png"
plt.savefig(out, dpi=300, bbox_inches="tight", facecolor=theme["bg"])
record_figure_output(out)
plt.show()
print(f"Saved -> {out}")
